<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Introduction to the `dts devel` API

The `dts devel` API is designed to work with a [`DTProject`](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/dtproject/introduction-to-duckietown-projects.html#). For a full overview of building and working with `DTProjects` to create new demos
[see this section in the Duckietown manual](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/dtproject/creating-demos.html).

The repo that we are going to use as our `DTProject` is the [`dt-ros2-core`](https://github.com/duckietown/dt-ros2-core), the ROS 2 port of the Duckietown autonomy stack, which you should have a copy of in the [packages](../packages/) folder if you followed the instructions about initializing the submodule in the [README](../README.md). You can notice that the `dt-ros2-core` repo has all of the elements of a `DTProject`, including:

 - the code in the [packages](../packages/dt-ros2-core/packages/) directory (inside `dt-ros2-core` - not to be confused with the `packages` directory in this learning experience): one ROS 2 package per component (`line_detector`, `ground_projection`, `lane_filter`, `lane_control`, `fsm`, `led_emitter`, ...), each with its `config/` and `launch/` directories, plus the `duckietown_demo` package that holds the demo launch files
 - the dependencies defined (`dependencies-apt.txt`, `dependencies-py3.txt`)
 - the [launchers](../packages/dt-ros2-core/launchers/) defined

The remainder of the instructions in this part will be run from the directory [packages/dt-ros2-core](../packages/dt-ros2-core/), which is the base directory of your `DTProject`. You can move into that directory with

```bash
    cd packages/dt-ros2-core
```

## The Duckiematrix

Similar to other learning experiences, we can run demos with `dts devel` on a real Duckiebot or on a virtual one inside the Duckiematrix.

The steps to get started with the Duckiematrix are described in [this section of the book](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/introduction-to-the-duckiematrix-virtual-environment.html).

In summary, the process is, depending on the environment you are using, one of the two below:

### The Duckiematrix on native Ubuntu system

If you are running Duckietown in a [native Ubuntu environment](https://docs.duckietown.com/ente/duckietown-manual/10-setup/00-computer/setup-ubuntu.html), then:

1. Run the matrix with a map that is appropriate for the demo you are trying to run:

```bash
    dts matrix run --standalone --embedded -m big_loop
```

2. In a new terminal, `create`, `start`, and `attach` a virtual robot (call it anything you want - here we use `ROBOT_NAME`) to the Duckiebot asset (in this case `map_0/vehicle_0`) that is inside the Duckiematrix

```bash
    dts duckiebot virtual create --type duckiebot --configuration DB21J ROBOT_NAME
    dts duckiebot virtual start ROBOT_NAME
    dts matrix attach ROBOT_NAME map_0/vehicle_0
```

### The Duckiematrix on macOS or Windows through Duckietown workspaces

If you are running Duckietown in a [workspace](https://docs.duckietown.com/ente/duckietown-manual/10-setup/00-computer/setup-duckietown-workspace.html?highlight=workspaces), then:

1. Inside the workspace: run the matrix engine using a map that is appropriate for the demo you are trying to run. In this case there is a map specified inside the project repo:

```bash
    dts matrix engine run --embedded -m big_loop
```

2. On the host machine, open a terminal and run:

```bash
    dts matrix run
```

3. In a new terminal inside the workspace, `create`, `start`, and `attach` a virtual robot (call it anything you want - here we use `ROBOT_NAME`) to the Duckiebot asset (in this case `map_0/vehicle_0`) that is inside the Duckiematrix:

```bash
    dts duckiebot virtual create --type duckiebot --configuration DB21J ROBOT_NAME
    dts duckiebot virtual start ROBOT_NAME
    dts matrix attach ROBOT_NAME map_0/vehicle_0
```

---

From this point on the virtual robot will be treated identically to a real robot.

## Bringing up the ROS 2 stack on the robot

The lane following demo talks to the robot through ROS 2 (Jazzy, over the zenoh middleware). The ROS 2 robot stack is started explicitly (virtual robots still autoboot the ROS 1 stack; a real Duckiebot should be updated first with `dts duckiebot update ROBOT_NAME`):

```bash
    dts stack up -H ROBOT_NAME ros2/duckiebot
    dts stack up -H ROBOT_NAME ros2-core/default
```

The first stack starts the ROS 2 drivers: `camera_node` (publishing `camera_node/image/compressed`), the wheel encoders (`left_wheel_encoder_driver_node/tick`, `right_wheel_encoder_driver_node/tick`), the wheels driver (`wheels_driver_node/wheels_cmd`) and the LEDs. The second one starts the *car interface* of `dt-ros2-core` (`launchers/default-duckiebot.sh` → `ros2 launch car_interface default.launch.py`): the `kinematics_node` that turns a `(v, omega)` command received on `car_cmd_switch_node/cmd` into wheel commands, the `car_cmd_switch_node` that decides which controller is allowed to drive, and the `joy_mapper_node` used by the keyboard control.

All topics live in the robot's namespace, e.g. `/ROBOT_NAME/camera_node/image/compressed`. You can check that everything is up from a terminal inside the code editor (`dts code editor`), which has the ROS 2 command line tools and connects to the robot once you run `dt-ros2-connect ROBOT_NAME` in it:

```bash
    ros2 node list
    ros2 topic list
    ros2 topic hz /ROBOT_NAME/camera_node/image/compressed
```

## Building the code

The next essential steps are building the code inside the `dt-ros2-core` package and running it on the Duckiebot. To build the code, make sure you are in the `packages/dt-ros2-core` directory and run:

    dts devel build -H ROBOT_NAME

where `ROBOT_NAME` is the name of your robot (real or virtual). The build runs `colcon build` on all the ROS 2 packages of `dt-ros2-core` inside a Docker image on the robot.

## Running the code

To run the code you will need to specify a launcher (or use the default one - named `default.sh` in the `launchers` directory). For the lane following demo, the launcher is called `lane-following.sh`, and all it does is:

```bash
    dt-exec ros2 launch duckietown_demos lane_following.launch.py veh:="${VEHICLE_NAME}"
```

We can run it using the command:

    dts devel run -H ROBOT_NAME -L lane-following

The launch file starts the `line_detector_node`, `ground_projection_node`, `lane_filter_node`, `lane_controller_node`, `fsm_node` and `led_emitter_node` in the namespace of your robot. You will know the command was executed successfully when the nodes show up in `ros2 node list` and the LEDs of your robot switch to the pattern of the initial FSM state (with the shipped configuration the robot boots directly into the `LANE_FOLLOWING` state, see below, so **place it in a lane before running the demo**).

## Other useful tips

It can be useful to look at the command line options for both `dts devel build` and `dts devel run` by adding the `--help` option. For our purposes the following two options with `dts devel run` may be useful.

### Forcing a build with local changes

The `-f` flag can be added to force the project to build even if there are local changes

### Running the code locally on your computer

You can build the code locally by omitting the `-H ROBOT_NAME`:

    dts devel build

and then run it and connect to your robot with

    dts devel run -R ROBOT_NAME

(the container then reaches the robot's zenoh router at `ROBOT_NAME.local`, exactly like the editor does).

### Omitting re-building for local changes that don't require recompiling

One of the advantages of Python is that it is a scripting language, so we do not always need to compile things. Given that this demo is completely implemented in Python and ROS 2, if the only changes you have made are modifications to parameters or code that does not require changes, there is actually no need to rerun the `dts devel build` command. Instead you can just sync your code with `-s` and mount it in inside the running container with `-M`:

    dts devel run -H ROBOT_NAME -L lane-following -s -M



# Running the lane following demo

When you ran `dts devel run` the finite state machine (`fsm_node`) started in its initial state. In the configuration shipped with `dt-ros2-core` ([`packages/fsm/config/fsm_node/lane_following.yaml`](../packages/dt-ros2-core/packages/fsm/config/fsm_node/lane_following.yaml)) this is `LANE_FOLLOWING`: the lane controller is enabled right away and the robot starts driving as soon as it sees the lane.

To take over manually (and give control back), run the `keyboard_control` in a new terminal:

    dts duckiebot keyboard_control ROBOT_NAME

With the keyboard control window in focus, you can press the `F` or toggle the `AutoPilot` switch. This publishes the joystick override on `joy_mapper_node/joystick_override`, which the FSM turns into the `NORMAL_JOYSTICK_CONTROL` / `LANE_FOLLOWING` transitions.

You can also set the FSM state directly from a terminal inside the code editor, by calling its `set_state` service:

```bash
    ros2 service call /ROBOT_NAME/fsm_node/set_state duckietown_msgs/srv/SetFSMState "{state: 'NORMAL_JOYSTICK_CONTROL'}"
    ros2 service call /ROBOT_NAME/fsm_node/set_state duckietown_msgs/srv/SetFSMState "{state: 'LANE_FOLLOWING'}"
```

and watch the current state with `ros2 topic echo /ROBOT_NAME/fsm_node/mode`.

## Tuning parameters

There is no central parameter server in ROS 2: every node declares its own parameters, with default values that come from the YAML files in the `config` folder of its package, e.g.

- [`packages/line_detector/config/line_detector_node/default.yaml`](../packages/dt-ros2-core/packages/line_detector/config/line_detector_node/default.yaml)
- [`packages/lane_filter/config/lane_filter_node/default.yaml`](../packages/dt-ros2-core/packages/lane_filter/config/lane_filter_node/default.yaml)
- [`packages/lane_control/config/lane_controller.yaml`](../packages/dt-ros2-core/packages/lane_control/config/lane_controller.yaml)
- [`packages/fsm/config/fsm_node/lane_following.yaml`](../packages/dt-ros2-core/packages/fsm/config/fsm_node/lane_following.yaml)

The demo launch file ([`packages/duckietown_demo/launch/lane_following.launch.py`](../packages/dt-ros2-core/packages/duckietown_demo/launch/lane_following.launch.py)) passes these files to the nodes. For the line detector and the lane filter it takes a `param_file` launch argument (`default.yaml` by default): you can add a new `ROBOT_NAME.yaml` file next to `default.yaml` in those two `config` folders and run the demo with `param_file:=ROBOT_NAME.yaml` appended to the `ros2 launch` line of `launchers/lane-following.sh`. Parameters in the YAML files are read at startup, so you need to restart the demo (`dts devel run ... -s -M` is enough, no rebuild) after changing them.

In most cases, you can also modify the parameters of a running node with the ROS 2 command line tools, from a terminal inside the code editor:

```bash
    ros2 param list /ROBOT_NAME/lane_controller_node
    ros2 param get  /ROBOT_NAME/lane_controller_node v_bar
    ros2 param set  /ROBOT_NAME/lane_controller_node v_bar 0.2
```

or with the parameter panels (`ParamPanel`) that you will find in the notebooks. However, these values will not persist if you restart the demo until you change the actual values that are loaded from the `config` folder of the node.


## An overview of how it works

The `Lane Following` demo consists of the following processing steps:


1. **Image capture**: an image is captured from your Duckiebot's camera (`camera_node/image/compressed`).
2. **Line detection**: colored line segments in the captured image are detected (`line_detector_node`, publishing `line_detector_node/segment_list`).
3. **Ground projection**: based on the camera's known intrinsic and extrinsic parameters, the detected line segments are projected onto the ground plane (`ground_projection_node`).
4. **Lane estimation**: the ground projected line segments and motion of your Duckiebot, which can be estimated with the help of its wheel encoders, are used to produce an estimate of your Duckiebot's lane pose (`lane_filter_node`, publishing `lane_filter_node/lane_pose`).
5. **Control**: based on the lane pose estimate, a controller sends control signals to adjust your Duckiebot's heading (`lane_controller_node`, publishing `lane_controller_node/car_cmd`).
6. **Actuation**: the control signals are applied to your Duckiebot's motors (`car_cmd_switch_node` → `kinematics_node` → `wheels_driver_node/wheels_cmd`).


In the remainder of the notebooks in this learning experience, we will go through the components of the lane following demo one by one, describe how they work, and what parameters could be tuned to get the lane following behavior to work better in your environment.

## A note on calibration

These instructions assume that you have fully [setup](https://docs.duckietown.com/ente/duckietown-manual/09-db-opmanual-intro/setup-duckiebot-intro.html) and [calibrated](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-calibrations-intro.html) your Duckiebot.


Let's proceed to the notebook [about the line detection](./01_line_detection.ipynb).
